<a href="https://colab.research.google.com/github/nshahi-data/bank-data-quality-eda/blob/main/01_data_loading_and_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports

In [3]:
import pandas as pd
import numpy as np


Read the uploaded CSV into a Dataframe


In [4]:
df = pd.read_csv("/content/cbd_bank_customers_sample.csv")
df.head()
df.info()
df.shape
df.describe()

df["Balance_AED"].mean()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Customer_ID            50 non-null     object 
 1   Customer_Name          50 non-null     object 
 2   Nationality            50 non-null     object 
 3   Branch                 50 non-null     object 
 4   Customer_Segment       50 non-null     object 
 5   Account_Type           50 non-null     object 
 6   Balance_AED            50 non-null     int64  
 7   Annual_Income_AED      50 non-null     int64  
 8   Credit_Score           49 non-null     float64
 9   KYC_Status             50 non-null     object 
 10  Phone_Number           49 non-null     float64
 11  Email                  50 non-null     object 
 12  Join_Date              50 non-null     object 
 13  Last_Transaction_Date  50 non-null     object 
dtypes: float64(2), int64(2), object(10)
memory usage: 5.6+ KB


np.float64(386436.2)

# view missing values + Basic Flags
# Credit_Score_Missing
# Phone_Number_Missing
# Email_Valid

In [5]:
Credit_Score_Missing = True

df.isna().sum()
df[df.isna().any(axis=1)]
df["Credit_Score_Missing"] = df["Credit_Score"].isna()

Phone_Number_Missing = True

df.isna().sum()
df[df.isna().any(axis=1)]
df["Phone_Number_Missing"] = df["Phone_Number"].isna()
df[["Phone_Number_Missing", "Phone_Number", "Phone_Number_Missing"]]
df["Customer_ID"].duplicated().sum()
df["Email"]
df["Email_Valid"] = df["Email"].str.match(
    r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
    na=False
)
df[df["Email_Valid"] == False][["Customer_ID", "Email"]]
df["Balance_AED"].describe()
df[df["Balance_AED"] < 0][["Customer_ID", "Balance_AED"]]
df[["Join_Date", "Last_Transaction_Date"]].dtypes



,0
Join_Date,object
Last_Transaction_Date,object


#  Data Type / Date Cleaning
#   date conversion
#   date validation

In [6]:
df["Join_Date"] = pd.to_datetime(
    df["Join_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Last_Transaction_Date"] = pd.to_datetime(
    df["Last_Transaction_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df[["Join_Date", "Last_Transaction_Date"]].isna().sum()

,0
Join_Date,0
Last_Transaction_Date,0


In [7]:
df["Join_Date"] = pd.to_datetime(df["Join_Date"], errors="coerce")
df[["Join_Date", "Last_Transaction_Date"]].isna().sum()
pd.to_datetime(..., errors="coerce")
raw_df = pd.read_csv("/content/cbd_bank_customers_sample.csv")
raw_df[["Join_Date", "Last_Transaction_Date"]].head(15)

,Join_Date,Last_Transaction_Date
0,09-09-2020,04-04-2021
1,19-12-2019,02-08-2020
2,11-09-2019,07-07-2020
3,27-04-2020,18-02-2021
4,23-04-2021,07-01-2022
5,14-05-2022,11-02-2023
6,06-11-2020,03-12-2021
7,25-10-2022,20-10-2023
8,06-09-2019,19-09-2020
9,24-04-2020,06-01-2021


In [8]:
df[["Join_Date", "Last_Transaction_Date"]].head()

,Join_Date,Last_Transaction_Date
0,2020-09-09,2021-04-04
1,2019-12-19,2020-08-02
2,2019-09-11,2020-07-07
3,2020-04-27,2021-02-18
4,2021-04-23,2022-01-07


Business Rules -
Business rules were applied separately from basic data validation to avoid treating valid but significant values as data errors.

- KYC Status — Pending and Expired records are flagged for follow-up.
- Low-Water Mark — balances below the defined threshold are flagged for attention rather than deleted or treated as invalid.
- Date Consistency — the last transaction date cannot precede the customer join date.
- Credit Score — missing scores are flagged for remediation; valid low scores are retained rather than treated as data errors.


In [9]:
df["Join_Date"] = pd.to_datetime(
    df["Join_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Last_Transaction_Date"] = pd.to_datetime(
    df["Last_Transaction_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df[["Join_Date", "Last_Transaction_Date"]].isna().sum()

,0
Join_Date,0
Last_Transaction_Date,0


In [10]:
df["Credit_Score"].describe()

,Credit_Score
count,49.000000
mean,713.612245
std,82.909041
min,589.000000
25%,646.000000
50%,703.000000
75%,793.000000
max,850.000000


In [11]:
df["Credit_Score_Valid"] = df["Credit_Score"].between(300, 900) | df["Credit_Score"].isna()

In [12]:
df[df["Credit_Score_Valid"] == False][["Customer_ID", "Credit_Score"]]

,Customer_ID,Credit_Score


In [13]:
df["Customer_ID"].isna().sum()

np.int64(0)

In [14]:
df["Customer_ID"].duplicated().sum()

np.int64(1)

In [15]:
df[df["Customer_ID"].duplicated(keep=False)]
df[df["Customer_ID"] == "CBD00035"].T

,34,35
Customer_ID,CBD00035,CBD00035
Customer_Name,Neha Sharma,Neha Sharma
Nationality,Philippines,Philippines
Branch,Jumeirah,Jumeirah
Customer_Segment,Retail,Retail
Account_Type,Salary,Salary
Balance_AED,23429,23429
Annual_Income_AED,888794,888794
Credit_Score,814.0,814.0
KYC_Status,Pending,Pending


In [16]:
df.loc[34].compare(df.loc[35])

,self,other


In [17]:
df = df.drop_duplicates()
df["Customer_ID"].duplicated().sum()

np.int64(0)

In [18]:
df["KYC_Status"].value_counts(dropna=False)

,count
KYC_Status,
Pending,22
Complete,14
Expired,13


In [19]:
df["KYC_Action_Required"] = df["KYC_Status"].isin(["Pending", "Expired"])
df[["KYC_Status", "KYC_Action_Required"]].value_counts()

,,count
KYC_Status,KYC_Action_Required,
Pending,True,22
Complete,False,14
Expired,True,13


In [20]:
df[df["Last_Transaction_Date"] < df["Join_Date"]][
    ["Customer_ID", "Join_Date", "Last_Transaction_Date"]
]

,Customer_ID,Join_Date,Last_Transaction_Date
24,CBD00025,2032-03-15,2023-02-28


In [21]:
df["Date_Valid"] = (
    df["Last_Transaction_Date"] >= df["Join_Date"]
)


In [22]:
df["Date_Valid"].value_counts()

,count
Date_Valid,
True,48
False,1


In [23]:
LOW_WATER_MARK = -2000

df["Low_Balance_Alert"] = df["Balance_AED"] < LOW_WATER_MARK

df[df["Low_Balance_Alert"]][
    ["Customer_ID", "Balance_AED"]
]

,Customer_ID,Balance_AED
4,CBD00005,-2500


## Data Quality Summary

The final summary consolidates the validation results after applying the data-quality and business rules. It highlights records requiring attention while preserving valid customer data wherever possible.


In [24]:
print("DATA QUALITY SUMMARY")
print("--------------------")
print("Total records:", len(df))
print("Missing Customer IDs:", df["Customer_ID"].isna().sum())
print("Duplicate Customer IDs:", df["Customer_ID"].duplicated().sum())
print("Invalid Credit Scores:", (~df["Credit_Score_Valid"]).sum())
print("KYC Action Required:", df["KYC_Action_Required"].sum())
print("Invalid Date Relationships:", (~df["Date_Valid"]).sum())
print("Low Balance Alerts:", df["Low_Balance_Alert"].sum())

DATA QUALITY SUMMARY
--------------------
Total records: 49
Missing Customer IDs: 0
Duplicate Customer IDs: 0
Invalid Credit Scores: 0
KYC Action Required: 35
Invalid Date Relationships: 1
Low Balance Alerts: 1
